# LTCM 

*Case: Long-Term Capital Management, L.P. (A) [9-200-007].*

# 1. READING

### 1. 
Describe LTCM’s investment strategy with regard to the following aspects:
* Securities traded
* Trading frequency
* Skewness (Do they seek many small wins or a few big hits?)
* Forecasting (What is behind their selection of trades?)

### 2. 
What are LTCM’s biggest advantages over its competitors?

### 3.
The case discusses four types of funding risk facing LTCM:
* collateral haircuts
* repo maturity
* equity redemption
* loan access

The case discusses specific ways in which LTCM manages each of these risks. Briefly discuss
them.

### 4. 

LTCM is largely in the business of selling liquidity and volatility. Describe how LTCM accounts
for liquidity risk in their quantitative measurements.

### 5.

Is leverage risk currently a concern for LTCM?

### 6. 

Many strategies of LTCM rely on converging spreads. LTCM feels that these are almost win/win
situations because of the fact that if the spread converges, they make money. If it diverges, the
trade becomes even more attractive, as convergence is still expected at a future date.

What is the risk in these convergence trades?

***

# 2. Fund Performance and Attribution

### Data

* `ltcm exhibits data.xlsx`, `Exhibit 2`: Gross and net (total) returns of LTCM
* `spy_data.xlsx`: SPY returns and risk-free rate (scaled tbill index)

### 1. Summary stats.

For both the gross and net series of LTCM excess returns, report the annualized 
* mean
* volatility
* Sharpe ratios

Also report the
* skewness
* kurtosis
* 5th quantile

### 2. Compare to SPY

Comment on how these stats compare to SPY and other assets we have seen. 

How much do they differ between gross and net?

In [155]:
import pandas as pd

returns=pd.read_excel('../data/ltcm_exhibits_data.xlsx',sheet_name=1)
columns=returns.iloc[1,1:].tolist()
returns=returns.dropna()
index=returns.iloc[:,0].tolist()
data=returns.iloc[:,1:].astype(float) 

returns=pd.DataFrame(data.values,index=index,columns=columns)
returns.index=returns.index.to_period('M')
returns.index.name = 'date' 


In [156]:
spy_excess=pd.read_excel('../data/spy_data.xlsx',sheet_name=3)
spy_excess=spy_excess.set_index('date')
spy_excess.index=pd.to_datetime(spy_excess.index).to_period('M')
rf=pd.read_excel('../data/spy_data.xlsx',sheet_name=1)
rf=rf[['date','^IRX']]
rf=rf.set_index('date')
rf.index=pd.to_datetime(rf.index).to_period('M')



In [157]:
df=returns.merge(spy_excess,on='date',how='inner')
df=df.merge(rf,on='date',how='inner')

In [158]:
columns=['Gross Monthly Performancea','Net Monthly Performanceb','SPY']

res=pd.DataFrame(index=columns,columns=['mean','vol','sharpe','skew','kurt','5th'])

for column in columns:

    res.loc[column,'mean']=df[column].mean()*12
    res.loc[column,'vol']=df[column].std()*12**0.5
    res.loc[column,'sharpe']=res.loc[column,'mean']/res.loc[column,'vol']
    res.loc[column,'skew']=df[column].skew()
    res.loc[column,'kurt']=df[column].kurtosis()
    res.loc[column,'5th']=df[column].quantile(0.05)


In [159]:
res

,mean,vol,sharpe,skew,kurt,5th
Gross Monthly Performancea,0.293887,0.136354,2.155321,-0.296428,1.569354,-0.0264
Net Monthly Performanceb,0.20717,0.111904,1.851315,-0.81787,2.905537,-0.0224
SPY,0.154775,0.114073,1.356806,-0.406867,-0.388002,-0.049667


### 3. LFD

Estimate a linear factor decomposition of **net** LTCM excess returns on `SPY` excess returns.

Report
* annualized alpha
* beta
* r-squared

Does LTCM deliver performance beyond `SPY`?

In [164]:
import statsmodels.api as sm

y=df['Net Monthly Performanceb']
x=sm.add_constant(df['SPY'])
reg=sm.OLS(y,x).fit()

res=pd.DataFrame(index=['Net Monthly Performanceb'],columns=['alpha','beta','r2'])

res.loc['Net Monthly Performanceb','alpha']=reg.params["const"] * 12
res.loc['Net Monthly Performanceb','beta']=reg.params["SPY"]
res.loc['Net Monthly Performanceb','r2']=reg.rsquared
res


,alpha,beta,r2
Net Monthly Performanceb,0.185039,0.14299,0.021246


$$\newcommand{\betalinear}{\beta_{\text{linear}}}
\newcommand{\betaquad}{\beta_{\text{quad}}}
$$

### 4. Nonlinear Exposure

Let's check for non-linear market exposure. Run the following regression on LTCM's **net** excess returns:

$$
\tilde{r}_t^{\text{ltcm}} = \alpha + \betalinear \tilde{r}_t^m + \betaquad \left(\tilde{r}_t^m\right)^2 + \epsilon_t
$$

Report 
* annualized alpha
* the linear and quadratic betas
* r-squared

In [171]:
df['quad']=df['SPY']**2
y=df['Net Monthly Performanceb']

x=sm.add_constant(df[['SPY','quad']])
reg=sm.OLS(y,x).fit()

res=pd.DataFrame(index=['Net Monthly Performanceb'],columns=['alpha','beta-linear','beta-quad','r2'])

res.loc['Net Monthly Performanceb','alpha']=reg.params["const"] * 12
res.loc['Net Monthly Performanceb','beta-linear']=reg.params["SPY"]
res.loc['Net Monthly Performanceb','beta-quad']=reg.params["quad"]
res.loc['Net Monthly Performanceb','r2']=reg.rsquared
res

,alpha,beta-linear,beta-quad,r2
Net Monthly Performanceb,0.212959,0.17121,-2.187023,0.028545


### 5. 

* Does the quadratic market factor do much to increase the overall LTCM variation explained by the market?
* From the regression evidence, does LTCM's market exposure behave as if it is long market options or short market options?
* Should we describe LTCM as being positively or negatively exposed to market volatility?

1. not much
2. short market options, since beta-quad is negative
3. negatively related 

### 6. 

Let's try to pinpoint the nature of LTCM's nonlinear exposure. Does it come more from exposure to up-markets or down-markets? Run the following regression on LTCM's net excess returns:

$$
\tilde{r}_t^{\text{ltcm}}  = \alpha + \beta\tilde{r}_t^m + \beta_u \max\left(\tilde{r}_t^m-k_1,0\right) + \beta_d \max\left(k_2 - \tilde{r}_t^m\right) + \epsilon_t
$$

where $k_1= .03$ and $k_2= -.03$. 

Report 
* annualized alpha
* market beta, the **up** and **down** betas
* r-squared

In [173]:
import numpy as np
df['up']=np.where(df['SPY'] > 0.03, df['SPY']-0.03,0)
df['down']=np.where(df['SPY'] < - 0.03, -0.03-df['SPY'], 0)
y=df['Net Monthly Performanceb']

x=sm.add_constant(df[['SPY','up','down']])
reg=sm.OLS(y,x).fit()

res=pd.DataFrame(index=['Net Monthly Performanceb'],columns=['alpha','beta','beta-up','beta-down','r2'])

res.loc['Net Monthly Performanceb','alpha']=reg.params["const"] * 12
res.loc['Net Monthly Performanceb','beta']=reg.params["SPY"]
res.loc['Net Monthly Performanceb','beta-up']=reg.params["up"]
res.loc['Net Monthly Performanceb','beta-down']=reg.params["down"]

res.loc['Net Monthly Performanceb','r2']=reg.rsquared
res

,alpha,beta,beta-up,beta-down,r2
Net Monthly Performanceb,0.159421,0.438662,-0.728775,1.046277,0.049557


### 7.

* Is LTCM long or short the call-like factor? And the put-like factor?
* Which factor moves LTCM more, the call-like factor, or the put-like factor?
* In the previous problem, you commented on whether LTCM is positively or negatively exposed to market volatility. Using this current regression, does this volatility exposure come more from being long the market's upside? Short the market's downside? Something else?

1. long put-like factor, short call-like facor
2. put-like factor
3. long downside vol, short upside vol 